# Interpretable MS/MS fragment–structure–property chain

This notebook executes the frozen, leakage-safe interpretation workflow for Supplementary Figure S4 and Tables S16–S19. It reuses the exact folds and artifacts from notebooks 05, 08, and 18. No external evaluation labels or manually generated attribution values are used.

The pilot completed successfully on 2026-10-11: all 45 frozen-OOF reproduction checks passed. The notebook is now configured for the final **full** run. Full mode remains programmatically gated by the successful pilot manifest.

In [ ]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / 'outputs' / 'pbt_msms_v1').exists():
    ROOT = ROOT.parent
if not (ROOT / 'outputs' / 'pbt_msms_v1').exists():
    ROOT = Path(r'E:\谱图预测PBT_终版')
assert (ROOT / 'outputs' / 'pbt_msms_v1').exists(), 'Project root not found.'

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pbt_msms.interpretability_chain as interpretability_chain
interpretability_chain = importlib.reload(interpretability_chain)
run_pipeline = interpretability_chain.run_pipeline
output_directory = interpretability_chain.output_directory

OUT = output_directory(ROOT)
print('ROOT:', ROOT)
print('Python:', sys.executable)
print('OUT:', OUT)

## Configuration

`RUN_MODE = 'full'` covers every eligible OOF compound, retains the top 40 peaks/bins, uses 32 Integrated Gradient steps, and executes compound-wide DreaMS peak-bin occlusion. The completed Pilot outputs are retained unchanged for audit.

In [ ]:
RUN_MODE = 'full'  # final manuscript analysis; Pilot gate already passed
RUN_DREAMS_PEAK_OCCLUSION = True
RUN_STAGES = ['binned', 'transformer', 'dreams', 'summarize']

print('Mode:', RUN_MODE)
print('Stages:', RUN_STAGES)
print('DreaMS compound-wide peak-bin occlusion:', RUN_DREAMS_PEAK_OCCLUSION)

## Execute

The pipeline first refits or reloads each frozen model and checks its predictions against the saved OOF results. Attribution is retained for audit, but the run is marked failed and full mode stays locked if reproduction QC fails.

In [ ]:
manifest = run_pipeline(
    ROOT,
    mode=RUN_MODE,
    run_dreams_peak_occlusion=RUN_DREAMS_PEAK_OCCLUSION,
    stages=RUN_STAGES,
)
display(pd.Series(manifest, name='value').to_frame())

## Quality-control review

In [ ]:
reproduction = pd.read_csv(OUT / f'model_reproduction_qc_{RUN_MODE}.csv')
selected = pd.read_csv(OUT / f'selected_compounds_{RUN_MODE}.csv')
attributions = pd.read_csv(OUT / f'attribution_records_{RUN_MODE}.csv')

display(reproduction.sort_values(['model', 'endpoint', 'outer_fold', 'seed']))
display(selected.groupby(['endpoint', 'model_family']).size().rename('n_selected').reset_index())
display(attributions.groupby(['endpoint_short', 'model', 'method']).size().rename('n_records').reset_index())

assert reproduction['passed'].astype(bool).all(), 'Frozen OOF reproduction QC failed.'
assert manifest['status'] == 'PASS', manifest
print('PASS: the next mode may be run after reviewing the tables above.')

## Output files

- `attribution_records_<mode>.csv`: peak, mass-bin, embedding-IG and attention records.
- `model_reproduction_qc_<mode>.csv`: frozen OOF reproduction audit.
- `selected_compounds_<mode>.csv`: deterministic endpoint/fold selection.
- `attribution_stability_<mode>.csv`: fold/seed concordance.
- `fragment_structure_property_links_<mode>.csv`: measured fragment–functional-group–endpoint associations.
- `negative_controls_<mode>.csv`: frozen label-permutation controls.
- `run_manifest_<mode>.json`: settings, status and output provenance.

In [ ]:
for name, path in manifest['outputs'].items():
    path = Path(path)
    print(f'{name:28s} {path.name:55s} {path.stat().st_size:,} bytes')